# 라이브러리 임포트 및 시드 고정


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
cd ../

h:\내 드라이브\Myway-person-classifier


In [2]:
pwd

'h:\\내 드라이브\\Myway-person-classifier'

In [1]:
import os
os.chdir('/content/drive/MyDrive/Myway-person-classifier')
print(f'현재 작업 디렉토리: {os.getcwd()}')


FileNotFoundError: [WinError 3] 지정된 경로를 찾을 수 없습니다: '/content/drive/MyDrive/Myway-person-classifier'

In [43]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [ ]:
%pip install -r ./requirements.txt --extra-index-url https://download.pytorch.org/whl/cu124


Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 113.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 102.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━ 502.4/664.8 MB 21.9 MB/s eta 0:00:08
ERROR: Operation cancelled by user


In [3]:
import pandas as pd
import numpy as np
import random
import re
import os
from sklearn.model_selection import train_test_split


In [4]:
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

SEED = 42
seed_everything(SEED)


# 데이터 불러오기


In [5]:
# 데이터 경로 설정
TRAIN_CSV = "./data/original_data/train.csv"
TEST_CSV = "./data/original_data/test.csv"
SUBMISSION_CSV = "./data/original_data/sample_submission.csv"

# 학습 데이터 불러오기
train_df = pd.read_csv(TRAIN_CSV, encoding="utf-8-sig")
test_df = pd.read_csv(TEST_CSV, encoding="utf-8-sig")
submission_df = pd.read_csv(SUBMISSION_CSV, encoding="utf-8-sig")
print("원본 학습 데이터 크기:", len(train_df))


원본 학습 데이터 크기: 97172


# TRAIN 데이터 전처리


In [6]:
def minimal_preprocess(text):
    text = text.strip()
    text = re.sub(r'[\u4E00-\u9FFF]', '', text)                   # 한자 제거
    text = re.sub(r'<[^>]+>', '', text)                           # HTML 태그 제거
    text = re.sub(r'\(\s*[^\w가-힣]*\s*\)', '', text)             # 빈 괄호 제거
    text = re.sub(r'\([^\(\)]{0,20}[\?\~]{1,3}[^\(\)]{0,20}\)', '', text)  # ( ? ~ ? ) 제거
    text = re.sub(r'[.,]{3,}', '.', text)                         # ... → .
    text = re.sub(r'[()]{2,}', '', text)                          # 괄호 잔재 정리
    text = re.sub(r',\s*,+', ',', text)
    text = re.sub(r'\s+', ' ', text)                              # 중복 공백 제거
    return text

def split_into_paragraphs(text):
    # 문단 기준: 두 줄 개행 우선
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    # 만약 너무 적게 쪼개졌으면 한 줄 개행으로 재시도
    if len(paragraphs) <= 1:
        paragraphs = [p.strip() for p in text.split('\n') if p.strip()]
    return paragraphs

def convert_train_to_paragraphs(train_df):
    rows = []
    for _, row in train_df.iterrows():
        title = row['title']
        full_text = row['full_text']  # 전처리 전 원본에서 문단 나눔
        label = row['generated']
        paragraphs = split_into_paragraphs(full_text)
        for idx, para in enumerate(paragraphs):
            cleaned_para = minimal_preprocess(para)  # 각 문단에 대해 전처리
            rows.append({
                'title': title,
                'paragraph_index': idx,
                'paragraph_text': cleaned_para,
                'generated': label
            })
    return pd.DataFrame(rows)


In [7]:
paragraph_train = convert_train_to_paragraphs(train_df)


In [8]:
# paragraph_text가 NaN인 행 개수 확인
nan_cnt = paragraph_train['paragraph_text'].isna().sum()
print(f"paragraph_text NaN 개수: {nan_cnt}")

# NaN 행 제거 및 인덱스 재정렬
paragraph_train = (
    paragraph_train
      .dropna(subset=['paragraph_text'])  # NaN 행 삭제
      .reset_index(drop=True)             # 인덱스 리셋
)

print("제거 후 데이터 크기:", len(paragraph_train))

paragraph_text NaN 개수: 0
제거 후 데이터 크기: 1226364


In [9]:
# ──────────────────────────────
# 데이터 크기를 1/4로 줄이기 (Stratified 샘플링)
# ──────────────────────────────
paragraph_train = paragraph_train.groupby('generated', group_keys=False).apply(
    lambda x: x.sample(frac=0.25, random_state=SEED)
).reset_index(drop=True)

print(f"1/4 샘플링 후 데이터 크기: {len(paragraph_train)}")
print(paragraph_train['generated'].value_counts())

1/4 샘플링 후 데이터 크기: 306591
generated
0    281413
1     25178
Name: count, dtype: int64


C:\Users\rjs72\AppData\Local\Temp\ipykernel_43780\3790581308.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  paragraph_train = paragraph_train.groupby('generated', group_keys=False).apply(


In [10]:
paragraph_train = paragraph_train.rename(columns={'paragraph_text': 'full_text'})


In [11]:
paragraph_train


,title,paragraph_index,full_text,generated
0,아르헨티나 축구 국가대표팀,33,디에고 마라도나의 임기.,0
1,2011년 태풍,84,'날개'는 조선민주주의인민공화국에서 제출한 이름이다.,0
2,성노예,7,약탈혼은 상대방을 납치하여 결혼하는 것이다. 남녀의 신체적 사회적 차이로 인하여 여...,0
3,김지훈 (권투 선수),3,2010년 5월 22일에는 미국 텍사스에서 벌어진 IBF 라이트급 챔피언 도전자 결...,0
4,X86 메모리 분할,5,"보통 명령어에 정해진 세그먼트가 기본으로 선택되지만, 사용하기를 원하는 세그먼트가 ...",0
...,...,...,...,...
306586,더베드의 왕 푸일,3,"얼마 뒤, 푸일과 그 밑의 조신들이 아르베르스(오늘날 펨브룩셔 나버스의 봉분 위에 ...",1
306587,잎,0,잎은 광합성과 증산작용 및 호흡 작용을 하는 식물의 기관 가운데 하나이다. 잎에서 ...,1
306588,강희맹,6,"강희맹은 《금양잡록》에서 ""경기에서는 동풍이 불 때 가뭄이 심해 어떤 해에는 논 밭...",1
306589,한비자,0,《한비자》는 전국 시대의 대표적인 법가 사상서입니다. 한비를 비롯한 여러 학자들의 ...,1


In [12]:
# 1) 문단 길이(문자 수) 계산
paragraph_train['char_len'] = paragraph_train['full_text'].str.len()

# 2) 라벨(generated)별 35 % · 95 % 퍼센타일 계산
percentiles = (
    paragraph_train
      .groupby('generated')['char_len']
      .quantile([0.35, 0.95])        # 두 지점 한 번에 구함
      .unstack(level=1)              # 보기 편하게: index=라벨, columns=p35/p95
      .rename(columns={0.35: 'p35', 0.95: 'p95'})
)

print("라벨별 문단 길이 퍼센타일")
print(percentiles)


라벨별 문단 길이 퍼센타일
             p35    p95
generated              
0          102.0  448.0
1          111.0  434.0


In [13]:
# 3) 위 기준을 이용해 필터링
mask = paragraph_train.apply(
    lambda r: percentiles.loc[r['generated'], 'p35'] <= r['char_len'] <= percentiles.loc[r['generated'], 'p95'],
    axis=1
)

filtered_df = (
    paragraph_train[mask]
      .reset_index(drop=True)
      .drop(columns=['char_len'])   # 길이 컬럼이 필요 없으면 제거
)

print(f"필터링 전: {len(paragraph_train)}  →  필터링 후: {len(filtered_df)}")


필터링 전: 306591  →  필터링 후: 184330


In [14]:
# 라벨별 개수 확인
label_counts = filtered_df['generated'].value_counts()
print("라벨별 개수 (필터링 후):")
print(label_counts)

# 1:1 언더샘플링 ─ 소수 클래스(1번)의 개수만큼만 0번에서 랜덤 추출
min_cnt = label_counts.min()            # 1번 라벨 개수
balanced_df = (
    filtered_df
      .groupby('generated', group_keys=False)
      .apply(lambda x: x.sample(n=min_cnt, random_state=SEED))
      .reset_index(drop=True)
)

print("\n언더샘플링 후 라벨별 개수:")
print(balanced_df['generated'].value_counts())

라벨별 개수 (필터링 후):
generated
0    169211
1     15119
Name: count, dtype: int64

언더샘플링 후 라벨별 개수:
generated
0    15119
1    15119
Name: count, dtype: int64


C:\Users\rjs72\AppData\Local\Temp\ipykernel_43780\2911492007.py:11: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min_cnt, random_state=SEED))


In [15]:
# Cell 21은 삭제 - Cell 15에서 이미 1/4 샘플링을 수행했으므로 중복 제거
# 이 셀은 비워둡니다


In [16]:
# Cell 22도 삭제 - 중복 코드 제거
# 이 셀은 비워둡니다

In [17]:
# Cell 23도 삭제 - 중복 print문 제거
# 이 셀은 비워둡니다

# Train/Validation 80/20 단일 분할


In [18]:
OUTPUT_DIR = "./train_simple/data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ──────────────────────────────
# Stratified Train/Validation 80/20 분할
# ──────────────────────────────
train_split, val_split = train_test_split(
    balanced_df,
    test_size=0.2,
    stratify=balanced_df['generated'],
    random_state=SEED
)

# 인덱스 리셋 및 셔플
train_split = train_split.sample(frac=1, random_state=SEED).reset_index(drop=True)
val_split = val_split.sample(frac=1, random_state=SEED).reset_index(drop=True)

print(f"Train 샘플 수: {len(train_split)}")
print(f"Validation 샘플 수: {len(val_split)}")
print(f"\nTrain 라벨 분포: {train_split['generated'].value_counts().to_dict()}")
print(f"Validation 라벨 분포: {val_split['generated'].value_counts().to_dict()}")


Train 샘플 수: 24190
Validation 샘플 수: 6048

Train 라벨 분포: {0: 12095, 1: 12095}
Validation 라벨 분포: {0: 3024, 1: 3024}


In [19]:
# ID 컬럼 추가
train_split.insert(0, 'id', [f"TRAIN_{i:05d}" for i in range(len(train_split))])
val_split.insert(0, 'id', [f"VAL_{i:05d}" for i in range(len(val_split))])

print("Train 첫 3개:")
print(train_split[['id', 'generated']].head(3))
print("\nValidation 첫 3개:")
print(val_split[['id', 'generated']].head(3))


Train 첫 3개:
            id  generated
0  TRAIN_00000          0
1  TRAIN_00001          0
2  TRAIN_00002          1

Validation 첫 3개:
          id  generated
0  VAL_00000          0
1  VAL_00001          1
2  VAL_00002          0


In [20]:
# CSV 저장
train_split.to_csv(os.path.join(OUTPUT_DIR, "train.csv"), index=False, encoding="utf-8-sig")
val_split.to_csv(os.path.join(OUTPUT_DIR, "val.csv"), index=False, encoding="utf-8-sig")

print(f"✓ train.csv → {os.path.join(OUTPUT_DIR, 'train.csv')} (행 {len(train_split)})")
print(f"✓ val.csv → {os.path.join(OUTPUT_DIR, 'val.csv')} (행 {len(val_split)})")


✓ train.csv → ./train_simple/data\train.csv (행 24190)
✓ val.csv → ./train_simple/data\val.csv (행 6048)


# TEST 데이터 전처리


In [21]:
test_df


,ID,title,paragraph_index,paragraph_text
0,TEST_0000,공중 도덕의 의의와 필요성,0,도덕이란 원래 개인의 자각에서 출발해 자기 의지로써 행동하는 일이다. 그러므로 도덕...
1,TEST_0001,공중 도덕의 의의와 필요성,1,도덕은 단순히 개인의 문제나 사회의 문제로 한정될 수 없다. 개인적인 측면과 사회적...
2,TEST_0002,공중 도덕의 의의와 필요성,2,"여기에 이른바 공중도덕은 실천적, 사회적 도덕의 한 부문이다. 즉, 공중 도덕이라 ..."
3,TEST_0003,공중 도덕의 의의와 필요성,3,우리가 공동 생활을 하는 데 있어서 공중 도덕이 필요함은 위에서 말한 것처럼 알 수...
4,TEST_0004,풍습과 그 개선,0,인간 사회에서는 다 함께 지켜야 할 어떤 기준이 있어 이를 따르면 옳다고 하고 따르...
...,...,...,...,...
1957,TEST_1957,저작권! 내가 먼저 지켜야지,11,"인터넷에는 음악뿐만 아니라 인터넷소설, 영화, 애니메이션 등 내가 좋아하는 것들이 ..."
1958,TEST_1958,저작권! 내가 먼저 지켜야지,12,하지만 이 경험을 통해 나는 달라진 시각을 갖게 되었다. 이제는 내가 좋아하는 콘텐...
1959,TEST_1959,저작권! 내가 먼저 지켜야지,13,그런데 누군가는 아무 노력이나 허락 없이 사용한다면 불공평하다. 우리나라는 인기 있...
1960,TEST_1960,저작권! 내가 먼저 지켜야지,14,"우리들이 우리나라의 노래, 영화, 드라마, 애니메이션 등의 저작권을 지켜 주어야 다..."


In [22]:
# paragraph_text에 전처리 적용
test_df['paragraph_text'] = test_df['paragraph_text'].apply(minimal_preprocess)


In [23]:
# 저장
test_df.to_csv(os.path.join(OUTPUT_DIR, "test_preprocessed.csv"), index=False, encoding='utf-8-sig')
print(f"✓ test_preprocessed.csv → {os.path.join(OUTPUT_DIR, 'test_preprocessed.csv')} (행 {len(test_df)})")


✓ test_preprocessed.csv → ./train_simple/data\test_preprocessed.csv (행 1962)


In [24]:
import shutil

shutil.copy('./data/original_data/sample_submission.csv', os.path.join(OUTPUT_DIR, 'sample_submission.csv'))
print(f"✓ sample_submission.csv 복사 완료")


✓ sample_submission.csv 복사 완료
